# Exercise 6: Apache Iceberg Time Travel — ClinVar Reclassification History
## BINFX410 — Chapter 10

---

## Learning Objectives
- Explain why mutable data (variant reclassifications) cannot be handled by traditional data lakes
- Create an Apache Iceberg table in Amazon Athena
- Perform DML operations (INSERT, UPDATE, DELETE) on an Iceberg table
- Demonstrate schema evolution (ADD COLUMN) without breaking existing queries
- Use time travel queries to reconstruct historical variant classifications
- Connect Iceberg capabilities to HIPAA audit trail requirements

**Estimated time:** 60 minutes  
**Dataset:** ClinVar summary variants (`s3://aws-roda-hcls-datalake/clinvar_summary_variants/`)  
**AWS services:** Athena (engine v3 required), S3, Glue Data Catalog

**Prerequisite:** Your Athena workgroup must be set to engine version 3. Check the Athena console → Workgroups → primary → Settings.

## Background: The ClinVar Reclassification Problem

### What is ClinVar?
ClinVar is NCBI's database of variant-disease associations. Clinical labs submit their interpretations of genomic variants, and these interpretations change over time as evidence accumulates:

```
2018: BRCA1 c.5266dupC → Pathogenic         (established)
2019: BRCA2 c.7480C>T  → Uncertain Significance
2021: BRCA2 c.7480C>T  → Pathogenic          (reclassified!)
2023: MLH1  c.350C>T   → Likely Benign
2024: MLH1  c.350C>T   → Benign              (upgraded certainty)
```

### The Traditional Data Lake Problem

On a plain S3 data lake, updating a variant means **overwriting the file**. This permanently destroys the history:

```python
# Traditional lake: overwrite destroys history
df.to_parquet('s3://my-bucket/clinvar.parquet')  # old version gone forever
```

This is clinically dangerous:
- A patient report from 2022 cited a variant as "Uncertain Significance"
- The variant was later reclassified as "Pathogenic"
- The lab needs to audit: what was the classification when the report was issued?
- With a traditional lake: impossible. The history is gone.

### The Iceberg Solution

Apache Iceberg maintains a **snapshot log** — every INSERT, UPDATE, DELETE creates a new snapshot. The old data is never deleted; new files are written. Time travel queries read from historical snapshots:

```sql
-- What was the classification on the day the patient report was issued?
SELECT clinicalsignificance
FROM clinvar_variants
FOR SYSTEM_TIME AS OF TIMESTAMP '2022-06-15 00:00:00'
WHERE name = 'NM_007294.3(BRCA1):c.5266dup'
```

## Section 1: Setup

**Important:** Iceberg DML (UPDATE/DELETE) requires Athena engine version 3.

**What this does:** This installs the required Python packages into the notebook environment. `boto3` is the AWS SDK, `awswrangler` is a higher-level wrapper for AWS data services, and `pandas` provides DataFrame support for local data manipulation before upload.

```python
!pip install boto3 awswrangler pandas --quiet
```

**What this does:** This initializes all AWS SDK clients (`boto3`) for S3, Athena, and an unsigned S3 client for public buckets, then verifies that the Athena workgroup is running engine version 3 — a hard requirement for Iceberg DML (UPDATE/DELETE). If the wrong engine version is detected, a warning is printed before any subsequent cells run.

```python
import boto3
import awswrangler as wr
import pandas as pd
import time
from datetime import datetime, timezone
from botocore import UNSIGNED
from botocore.config import Config

STUDENT_ID = "sab032226"  # CHANGE THIS
AWS_REGION = "us-east-1"
BUCKET = f"binfx410-datalake-{STUDENT_ID}"
ATHENA_RESULTS = f"s3://binfx410-athena-results-{STUDENT_ID}/"

s3 = boto3.client('s3', region_name=AWS_REGION)
s3_unsigned = boto3.client('s3', region_name=AWS_REGION,
                            config=Config(signature_version=UNSIGNED))
athena = boto3.client('athena', region_name=AWS_REGION)
session = boto3.Session(region_name=AWS_REGION)

# Verify Athena engine version
wg_response = athena.get_work_group(WorkGroup='primary')
engine = wg_response['WorkGroup']['Configuration'].get('EngineVersion', {}).get('SelectedEngineVersion', 'unknown')
print(f"Athena Engine Version: {engine}")
if 'version 3' not in engine.lower() and '3' not in engine:
    print("WARNING: Iceberg DML requires Athena engine version 3!")
    print("Update via: AWS Console → Athena → Workgroups → primary → Edit → Engine version 3")
else:
    print("Engine version 3 confirmed. Iceberg DML is supported.")
```

## Section 2: Creating the ClinVar Iceberg Table

**What this does:** This defines a reusable Python helper that submits SQL to the Athena API, polls for completion, and raises an exception on failure. All subsequent DDL and DML operations in this notebook (CREATE TABLE, INSERT, UPDATE, DELETE, ALTER TABLE) are executed through this function, which handles the asynchronous Athena execution model.

```python
# Helper to run DDL/DML in Athena and wait for completion
def run_athena_sql(sql, database='binfx410_gold', return_results=False):
    """Execute SQL in Athena engine v3 and wait for completion."""
    response = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={'Database': database},
        ResultConfiguration={'OutputLocation': ATHENA_RESULTS},
        WorkGroup='primary'
    )
    qid = response['QueryExecutionId']
    
    for _ in range(60):  # wait up to 2 minutes
        time.sleep(2)
        result = athena.get_query_execution(QueryExecutionId=qid)
        state = result['QueryExecution']['Status']['State']
        if state in ('SUCCEEDED', 'FAILED', 'CANCELLED'):
            if state == 'FAILED':
                reason = result['QueryExecution']['Status'].get('StateChangeReason', '')
                raise Exception(f"Query failed: {reason}")
            break
    
    if return_results:
        return wr.athena.read_sql_query(
            sql=sql, database=database,
            s3_output=ATHENA_RESULTS, boto3_session=session
        )
    return qid

print("SQL helper defined.")
```

**What this does:** This downloads the first 500 rows of the ClinVar `variant_summary.txt.gz` file directly from the NCBI FTP server over HTTPS, normalizes the column names to lowercase, selects the 11 columns this notebook needs, and coerces `variationid` to a clean integer. The result is a local pandas DataFrame that will be uploaded to S3 in the next cell.

```python
# The aws-roda-hcls-datalake bucket is no longer publicly accessible.
# Download ClinVar variant_summary directly from NCBI — same data, authoritative source.
CLINVAR_NCBI_URL = "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz"

print("Downloading ClinVar variant_summary from NCBI (first 500 rows)...")

df_raw = pd.read_csv(
    CLINVAR_NCBI_URL,
    sep='\t',
    compression='gzip',
    low_memory=False,
    nrows=500
)

# Normalize column names: lowercase and strip leading # (NCBI uses #AlleleID as first col)
df_raw.columns = [c.lower().lstrip('#') for c in df_raw.columns]

# Select the columns the rest of the notebook expects
NEEDED = ['variationid', 'name', 'geneid', 'genesymbol', 'clinicalsignificance',
          'reviewstatus', 'chromosome', 'start', 'stop', 'referenceallele', 'alternateallele']
df_clinvar = df_raw[NEEDED].copy()

# Ensure variationid is a clean integer (required for Athena bigint and UPDATE/DELETE)
df_clinvar['variationid'] = pd.to_numeric(df_clinvar['variationid'], errors='coerce')
df_clinvar = df_clinvar.dropna(subset=['variationid'])
df_clinvar['variationid'] = df_clinvar['variationid'].astype('int64')

print(f"Loaded {len(df_clinvar)} ClinVar variants")
print("\nClinical significance distribution:")
print(df_clinvar['clinicalsignificance'].value_counts().head(8).to_string())
```

**What this does:** This uploads the local ClinVar DataFrame to your S3 bucket as a single Parquet file at the path `s3://{BUCKET}/gold/clinvar_seed/clinvar_seed.parquet`. This staging file serves as the data source for the subsequent Athena `INSERT INTO` statement that populates the Iceberg table.

```python
# Write seed data to S3 (Iceberg needs data in S3 before we INSERT)
wr.s3.to_parquet(
    df=df_clinvar,
    path=f"s3://{BUCKET}/gold/clinvar_seed/clinvar_seed.parquet",
    dataset=False,
    boto3_session=session
)
print(f"Seed data written to s3://{BUCKET}/gold/clinvar_seed/clinvar_seed.parquet")
```

**What this does:** This creates an Apache Iceberg table via Athena engine v3, storing metadata in the Glue Data Catalog and writing data files in Parquet/Snappy format to `s3://{BUCKET}/gold/clinvar_iceberg/`. The `TBLPROPERTIES ('table_type' = 'ICEBERG')` flag activates full ACID transaction support, snapshot tracking, and schema evolution. Any existing table and its S3 data are dropped first to ensure a clean state on re-runs.

```python
try:
    glue = boto3.client('glue', region_name=AWS_REGION)
    glue.create_database(DatabaseInput={'Name': 'binfx410_gold'})
except: pass

# Drop and recreate to ensure a clean state on re-runs.
# 'CREATE TABLE IF NOT EXISTS' would silently keep an old table that may have
# extra columns (e.g. last_reviewed_date added by cell-16 in a previous run),
# causing the INSERT below to fail with a column-count mismatch.
run_athena_sql("DROP TABLE IF EXISTS binfx410_gold.clinvar_iceberg")
wr.s3.delete_objects(
    path=f"s3://{BUCKET}/gold/clinvar_iceberg/",
    boto3_session=session
)

# Athena Iceberg tables require 'string' — varchar(n) is not supported in DDL
create_iceberg_table = f"""
CREATE TABLE binfx410_gold.clinvar_iceberg (
    variationid          bigint,
    name                 string,
    geneid               string,
    genesymbol           string,
    clinicalsignificance string,
    reviewstatus         string,
    chromosome           string,
    start                bigint,
    stop                 bigint,
    referenceallele      string,
    alternateallele      string
)
LOCATION 's3://{BUCKET}/gold/clinvar_iceberg/'
TBLPROPERTIES (
    'table_type' = 'ICEBERG',
    'format' = 'parquet',
    'write_compression' = 'snappy'
)
"""

run_athena_sql(create_iceberg_table)
print("Iceberg table created: binfx410_gold.clinvar_iceberg")
print("This is backed by S3 with full snapshot tracking enabled.")
```

**What this does:** This registers the seed Parquet file as a Glue catalog table (`binfx410_gold.clinvar_seed`), then issues an Athena `INSERT INTO ... SELECT` statement that reads from the seed table and writes the initial set of ClinVar variants into the Iceberg table. This first insert creates Snapshot 0 in the Iceberg snapshot log — the baseline for all future time travel queries.

```python
# Register the seed Parquet as a Glue table so Athena can SELECT from it
wr.s3.to_parquet(
    df=df_clinvar,
    path=f"s3://{BUCKET}/gold/clinvar_seed/",
    dataset=True,
    database='binfx410_gold',
    table='clinvar_seed',
    boto3_session=session,
    mode='overwrite'
)

# Insert seed data into the Iceberg table.
# Note the type asymmetry in Athena:
#   CREATE TABLE DDL  → use 'string'   (Iceberg/Glue catalog type)
#   CAST in DML/SELECT → use 'varchar' (Presto/Trino SQL type)
insert_sql = f"""
INSERT INTO binfx410_gold.clinvar_iceberg
SELECT
    CAST(variationid AS bigint),
    CAST(name AS varchar),
    CAST(geneid AS varchar),
    CAST(genesymbol AS varchar),
    CAST(clinicalsignificance AS varchar),
    CAST(reviewstatus AS varchar),
    CAST(chromosome AS varchar),
    CAST(start AS bigint),
    CAST(stop AS bigint),
    CAST(referenceallele AS varchar),
    CAST(alternateallele AS varchar)
FROM binfx410_gold.clinvar_seed
"""

run_athena_sql(insert_sql)

snapshot_t0 = datetime.now(timezone.utc)
print(f"Initial INSERT complete at: {snapshot_t0.strftime('%Y-%m-%d %H:%M:%S UTC')}")

count_df = wr.athena.read_sql_query(
    sql="SELECT COUNT(*) as n FROM binfx410_gold.clinvar_iceberg",
    database='binfx410_gold',
    s3_output=ATHENA_RESULTS,
    boto3_session=session
)
print(f"Rows in Iceberg table: {count_df['n'].iloc[0]}")
```

## Section 3: Simulating a Reclassification Event

**What this does:** This runs an Athena `SELECT` query against the live Iceberg table to find up to 5 variants currently classified as "Uncertain Significance" (VUS). These variant IDs will be used as the targets of the simulated reclassification UPDATE in the next cell.

```python
# Check current classifications — find some VUS candidates to reclassify
vus_df = wr.athena.read_sql_query(
    sql="""
    SELECT variationid, name, genesymbol, clinicalsignificance
    FROM binfx410_gold.clinvar_iceberg
    WHERE clinicalsignificance LIKE '%Uncertain%'
    LIMIT 5
    """,
    database='binfx410_gold',
    s3_output=ATHENA_RESULTS,
    boto3_session=session
)
print("Variants of Uncertain Significance (VUS) — candidates for reclassification:")
vus_df
```

**What this does:** This records a wall-clock timestamp immediately before issuing an Athena `UPDATE` statement that changes up to 3 VUS variants to "Pathogenic", simulating a real ClinVar reclassification event. Iceberg writes new data files for the changed rows and registers a new snapshot — the original VUS classifications are preserved in the previous snapshot and remain queryable via time travel. The before/after timestamps are saved for use in the time travel queries in Section 6.

```python
# Record time BEFORE reclassification (the "past" snapshot)
time.sleep(2)  # ensure distinct timestamp
before_reclassification = datetime.now(timezone.utc)
print(f"Recording 'before' timestamp: {before_reclassification.strftime('%Y-%m-%d %H:%M:%S UTC')}")
time.sleep(2)  # ensure the UPDATE is after this timestamp

# Reclassify VUS variants as Pathogenic (simulate new evidence)
# In real ClinVar, this happens when labs submit additional supporting evidence
if len(vus_df) > 0:
    vus_ids = vus_df['variationid'].tolist()[:3]  # reclassify first 3
    ids_str = ', '.join(str(v) for v in vus_ids)
    
    update_sql = f"""
    UPDATE binfx410_gold.clinvar_iceberg
    SET clinicalsignificance = 'Pathogenic',
        reviewstatus = 'criteria provided, multiple submitters, no conflicts'
    WHERE variationid IN ({ids_str})
    """
    
    run_athena_sql(update_sql)
    after_reclassification = datetime.now(timezone.utc)
    print(f"\nReclassified {len(vus_ids)} variants from VUS → Pathogenic")
    print(f"Reclassification timestamp: {after_reclassification.strftime('%Y-%m-%d %H:%M:%S UTC')}")
    print(f"Variant IDs reclassified: {vus_ids}")
else:
    print("No VUS variants found. The dataset may not contain Uncertain Significance variants.")
    before_reclassification = datetime.now(timezone.utc)
    after_reclassification = datetime.now(timezone.utc)
    vus_ids = []
```

**What this does:** This queries the current (post-UPDATE) state of the Iceberg table to confirm that the reclassified variant IDs now show "Pathogenic" as their clinical significance. This is a verification step against the live snapshot — the same rows will show "Uncertain Significance" when queried via time travel using the `before_reclassification` timestamp.

```python
# Verify the UPDATE took effect
if vus_ids:
    updated_df = wr.athena.read_sql_query(
        sql=f"""
        SELECT variationid, name, genesymbol, clinicalsignificance, reviewstatus
        FROM binfx410_gold.clinvar_iceberg
        WHERE variationid IN ({', '.join(str(v) for v in vus_ids)})
        """,
        database='binfx410_gold',
        s3_output=ATHENA_RESULTS,
        boto3_session=session
    )
    print("After reclassification — current state:")
    updated_df
```

## Section 4: Schema Evolution — Adding a Column

**What this does:** This runs an Athena `ALTER TABLE ... ADD COLUMNS` statement that appends a new `last_reviewed_date` column to the Iceberg table schema. Because Iceberg handles schema evolution through metadata updates only, no existing Parquet data files are rewritten — existing rows will simply return `NULL` for the new column when queried. This demonstrates a key Iceberg advantage over traditional Hive-style tables where schema changes often require full table rewrites.

```python
# Add a new column: last_reviewed_date
# This simulates the ClinVar practice of tracking when each variant was last reviewed
# Iceberg handles this WITHOUT rewriting existing data files

add_column_sql = """
ALTER TABLE binfx410_gold.clinvar_iceberg
ADD COLUMNS (last_reviewed_date date)
"""

run_athena_sql(add_column_sql)
print("Column 'last_reviewed_date' added to clinvar_iceberg.")
print()
print("Key point: existing rows still have last_reviewed_date = NULL")
print("No data rewrite occurred — Iceberg handles schema evolution via metadata only.")
```

**What this does:** This queries the Iceberg table to confirm that pre-existing rows read correctly after the schema change (returning `NULL` for `last_reviewed_date`), then issues a second `UPDATE` statement to populate the new column for the previously reclassified variants. This demonstrates that new columns are immediately writable and that old rows remain queryable without any migration step.

```python
# Verify old rows still queryable with new schema (NULL for new column)
schema_check = wr.athena.read_sql_query(
    sql="""
    SELECT variationid, genesymbol, clinicalsignificance, last_reviewed_date
    FROM binfx410_gold.clinvar_iceberg
    LIMIT 5
    """,
    database='binfx410_gold',
    s3_output=ATHENA_RESULTS,
    boto3_session=session
)
print("Old rows with new column (NULL for pre-existing rows):")
print(schema_check.to_string(index=False))
print()
print("Now update the reclassified variants with a review date...")

if vus_ids:
    run_athena_sql(f"""
    UPDATE binfx410_gold.clinvar_iceberg
    SET last_reviewed_date = DATE '2026-03-15'
    WHERE variationid IN ({', '.join(str(v) for v in vus_ids)})
    """)
    print("Review dates set for reclassified variants.")
```

## Section 5: Simulating a Submission Retraction

**What this does:** This identifies up to 3 ClinVar variants with `reviewstatus = 'no assertion criteria provided'` (the weakest evidence tier) and issues an Athena `DELETE` statement to remove them, simulating a ClinVar submission retraction. Iceberg writes a new snapshot that excludes the deleted rows — the rows are not physically removed from S3 and remain accessible via time travel using the `before_retraction` timestamp recorded just before the DELETE.

```python
# ClinVar periodically retracts submissions from labs that fail to resubmit
# Simulate deleting variants from a specific submitter/review status

# Find some records to delete
to_delete = wr.athena.read_sql_query(
    sql="""
    SELECT variationid, name, genesymbol, reviewstatus
    FROM binfx410_gold.clinvar_iceberg
    WHERE reviewstatus = 'no assertion criteria provided'
    LIMIT 3
    """,
    database='binfx410_gold',
    s3_output=ATHENA_RESULTS,
    boto3_session=session
)

print("Submissions to be retracted (no assertion criteria provided):")
print(to_delete.to_string(index=False))

time.sleep(2)
before_retraction = datetime.now(timezone.utc)
time.sleep(2)

if len(to_delete) > 0:
    del_ids = to_delete['variationid'].tolist()
    del_ids_str = ', '.join(str(v) for v in del_ids)
    
    run_athena_sql(f"""
    DELETE FROM binfx410_gold.clinvar_iceberg
    WHERE variationid IN ({del_ids_str})
    """)
    
    after_retraction = datetime.now(timezone.utc)
    print(f"\nDeleted {len(del_ids)} retracted submissions.")

    # Verify deletion
    remaining = wr.athena.read_sql_query(
        sql=f"SELECT COUNT(*) as n FROM binfx410_gold.clinvar_iceberg",
        database='binfx410_gold', s3_output=ATHENA_RESULTS, boto3_session=session
    )
    print(f"Rows remaining: {remaining['n'].iloc[0]}")
else:
    print("No records with 'no assertion criteria provided' found.")
    before_retraction = datetime.now(timezone.utc)
    after_retraction = datetime.now(timezone.utc)
    del_ids = []
```

## Section 6: Time Travel Queries

**What this does:** This formats the two wall-clock timestamps recorded around the reclassification UPDATE into the `YYYY-MM-DD HH:MM:SS` string format required by Athena's `FOR TIMESTAMP AS OF TIMESTAMP '...'` time travel syntax. These strings are used as literals in the Athena SQL issued in the next two cells.

```python
# FORMAT the timestamps for Athena time travel syntax
before_ts = before_reclassification.strftime('%Y-%m-%d %H:%M:%S')
after_ts = after_reclassification.strftime('%Y-%m-%d %H:%M:%S')

print(f"Time travel comparison:")
print(f"  Before reclassification: {before_ts}")
print(f"  After reclassification:  {after_ts}")
```

**What this does:** These two Athena queries use Iceberg's `FOR TIMESTAMP AS OF TIMESTAMP '...'` syntax to read the table as it existed at a specific point in time, without modifying any data. The first query reads from the snapshot that existed before the reclassification UPDATE, showing "Uncertain Significance" for the targeted variants. The second query reads the current snapshot, showing "Pathogenic". Comparing the two results demonstrates the complete reclassification audit trail that Iceberg maintains automatically.

```python
# Time travel query: what were the classifications BEFORE the reclassification event?
# Athena Iceberg time travel syntax:
#   FOR TIMESTAMP AS OF TIMESTAMP '...'   ← time-based  (used here)
#   FOR VERSION AS OF <snapshot_id>       ← snapshot-based
# Note: NOT 'FOR SYSTEM_TIME AS OF' — that is ANSI SQL standard but Athena uses the shorter form.
if vus_ids:
    time_travel_before = wr.athena.read_sql_query(
        sql=f"""
        SELECT variationid, genesymbol, clinicalsignificance, reviewstatus
        FROM binfx410_gold.clinvar_iceberg
        FOR TIMESTAMP AS OF TIMESTAMP '{before_ts}'
        WHERE variationid IN ({', '.join(str(v) for v in vus_ids)})
        """,
        database='binfx410_gold',
        s3_output=ATHENA_RESULTS,
        boto3_session=session,
        ctas_approach=False
    )

    time_travel_after = wr.athena.read_sql_query(
        sql=f"""
        SELECT variationid, genesymbol, clinicalsignificance, reviewstatus
        FROM binfx410_gold.clinvar_iceberg
        WHERE variationid IN ({', '.join(str(v) for v in vus_ids)})
        """,
        database='binfx410_gold',
        s3_output=ATHENA_RESULTS,
        boto3_session=session
    )

    print(f"=== BEFORE reclassification ({before_ts}) ===")
    print(time_travel_before.to_string(index=False))
    print()
    print(f"=== CURRENT (after reclassification) ===")
    print(time_travel_after.to_string(index=False))
    print()
    print("Time travel confirmed: historical classifications are preserved!")
else:
    print("Skipped — no VUS variants were reclassified.")
```

**What this does:** This Athena time travel query reads the Iceberg table as it existed at the `before_retraction` timestamp to recover rows that were subsequently deleted by the submission retraction DELETE statement. The deleted rows are not present in the current snapshot but remain physically in S3 under the Iceberg data directory, accessible only through the snapshot log. This demonstrates how Iceberg snapshots satisfy the requirement to recover retracted data for audit purposes.

```python
# Time travel to recover deleted records
if del_ids:
    before_del_ts = before_retraction.strftime('%Y-%m-%d %H:%M:%S')

    recovered = wr.athena.read_sql_query(
        sql=f"""
        SELECT variationid, name, genesymbol, reviewstatus
        FROM binfx410_gold.clinvar_iceberg
        FOR TIMESTAMP AS OF TIMESTAMP '{before_del_ts}'
        WHERE variationid IN ({', '.join(str(v) for v in del_ids)})
        """,
        database='binfx410_gold',
        s3_output=ATHENA_RESULTS,
        boto3_session=session,
        ctas_approach=False
    )
    print(f"Deleted records recovered via time travel (as of {before_del_ts}):")
    print(recovered.to_string(index=False))
    print()
    print("These records are gone from the current view but preserved in Iceberg snapshots.")
```

## Section 7: Inspecting the Iceberg Snapshot History

**What this does:** This queries the Iceberg virtual metadata table `clinvar_iceberg$snapshots` via Athena to retrieve the complete snapshot history of the table, including the snapshot ID, parent snapshot ID, commit timestamp, operation type (append, overwrite, delete), and a summary of rows added/deleted per operation. This snapshot log is the foundation of Iceberg's time travel capability and can serve as a HIPAA-compliant audit trail showing every mutation made to the dataset.

```python
# Iceberg maintains a metadata table of all snapshots.
# The $snapshots virtual table also requires ctas_approach=False.
snapshots = wr.athena.read_sql_query(
    sql="""
    SELECT
        snapshot_id,
        parent_id,
        committed_at,
        operation,
        summary
    FROM binfx410_gold.\"clinvar_iceberg$snapshots\"
    ORDER BY committed_at
    """,
    database='binfx410_gold',
    s3_output=ATHENA_RESULTS,
    boto3_session=session,
    ctas_approach=False
)

print("Iceberg snapshot history:")
print(f"Total snapshots: {len(snapshots)}")
for _, row in snapshots.iterrows():
    print(f"\n  Snapshot: {row['snapshot_id']}")
    print(f"  Time:     {row['committed_at']}")
    print(f"  Operation:{row['operation']}")
    print(f"  Summary:  {row['summary']}")
```

## Section 8: Traditional Lake vs Iceberg vs RDBMS Comparison

**What this does:** This builds a local pandas DataFrame summarizing the feature differences between a traditional S3 data lake, an Apache Iceberg table on S3, and a relational database (PostgreSQL) across dimensions relevant to clinical genomics — ACID support, time travel, schema evolution, scale, storage cost, HIPAA audit readiness, and suitability for ClinVar-style mutable data. No AWS resources are created; this cell runs entirely locally and outputs the comparison as a formatted table.

```python
comparison = pd.DataFrame([
    {
        'Feature': 'ACID transactions (UPDATE/DELETE)',
        'Traditional S3 Lake': 'No — overwrite only',
        'Iceberg on S3': 'Yes — full ACID',
        'RDBMS (PostgreSQL)': 'Yes'
    },
    {
        'Feature': 'Time travel / version history',
        'Traditional S3 Lake': 'No',
        'Iceberg on S3': 'Yes — snapshot-based',
        'RDBMS (PostgreSQL)': 'Only with audit log tables'
    },
    {
        'Feature': 'Schema evolution (add column)',
        'Traditional S3 Lake': 'Manual table recreation',
        'Iceberg on S3': 'Yes — metadata only, no rewrite',
        'RDBMS (PostgreSQL)': 'Yes — ALTER TABLE'
    },
    {
        'Feature': 'Scale (TB+)',
        'Traditional S3 Lake': 'Yes — S3 unlimited',
        'Iceberg on S3': 'Yes — S3 unlimited',
        'RDBMS (PostgreSQL)': 'Limited — disk-bound'
    },
    {
        'Feature': 'Storage cost (per GB/month)',
        'Traditional S3 Lake': '$0.023',
        'Iceberg on S3': '$0.023 + ~10-20% overhead for old snapshots',
        'RDBMS (PostgreSQL)': '$0.115 (EBS gp2)'
    },
    {
        'Feature': 'HIPAA audit trail readiness',
        'Traditional S3 Lake': 'Manual — S3 versioning only',
        'Iceberg on S3': 'Yes — snapshot log is the audit trail',
        'RDBMS (PostgreSQL)': 'Requires trigger-based audit tables'
    },
    {
        'Feature': 'Appropriate for ClinVar history',
        'Traditional S3 Lake': 'No',
        'Iceberg on S3': 'Yes',
        'RDBMS (PostgreSQL)': 'Yes, but costly at scale'
    }
])

comparison.set_index('Feature', inplace=True)
comparison
```

## Exercise: YOUR CODE HERE

**What this does:** When implemented, this would issue an Athena `UPDATE` statement to reclassify all "Likely pathogenic" variants to "Pathogenic", record wall-clock timestamps before and after, and then run a `FOR TIMESTAMP AS OF` time travel query to confirm that the "Likely pathogenic" classifications are visible in the historical snapshot. This exercises the same reclassification workflow used in Section 3 but targets a different starting classification tier.

```python
# TASK 1: Reclassify all 'Likely pathogenic' variants to 'Pathogenic'
# Record the timestamp before and after
# Then use time travel to confirm the 'Likely pathogenic' classifications existed before

# YOUR CODE HERE
```

**What this does:** When implemented, this would issue an Athena `ALTER TABLE ... ADD COLUMNS` to add a `submitter_count` integer column, then run an `UPDATE` to populate it with random values for 10 rows, and finally query the table to confirm that rows not targeted by the UPDATE still return `NULL` for `submitter_count`. This reinforces the schema evolution concept from Section 4 — adding a column is a metadata-only operation that does not rewrite existing data files.

```python
# TASK 2: Add a second new column: submitter_count (integer)
# Update 10 random variants with submitter_count = a random number between 1 and 20
# Verify old rows still return NULL for submitter_count

# YOUR CODE HERE
```

**What this does:** When implemented, this would run an Athena query that joins the current Iceberg table to a time travel version of the same table (using `FOR TIMESTAMP AS OF`) and counts rows where `clinicalsignificance` differs between the two snapshots. This self-join across time produces a precise count of how many variant classifications have changed since the initial data load, demonstrating how Iceberg snapshots can power longitudinal audit reports.

```python
# TASK 3: Write a time travel query that counts how many variants
# changed classification between the initial load and now
# (Hint: join the current table to a time travel version and compare clinicalsignificance)

# YOUR CODE HERE
reclassification_count_query = """
-- YOUR SQL HERE
-- Count rows where clinicalsignificance differs between the initial snapshot and now
"""
```

## Reflection Questions

*(Double-click to edit)*

1. **Iceberg stores old data files even after UPDATE/DELETE.** This means storage accumulates over time. How would you decide when to run `OPTIMIZE` (Iceberg's compaction command) and `VACUUM` (delete old snapshots) on a ClinVar table? What clinical governance consideration might prevent you from vacuuming too aggressively?

2. **The ClinVar submission process allows many labs to submit interpretations for the same variant.** If 5 labs each submit different classifications for variant rs80357906, and one lab later retracts their submission, how should this affect the `clinicalsignificance` column? Is a single `UPDATE` statement sufficient?

3. **HIPAA requires an audit trail for access to PHI** (Protected Health Information). ClinVar is public, but an internal clinical lab's variant database linking variants to specific patients is PHI. How would Iceberg's snapshot history serve as a HIPAA-compliant audit trail? What would it NOT cover that a traditional audit log would?

4. **Schema evolution is a silent risk in data pipelines.** If a downstream analysis script reads `clinicalsignificance` as column index 5 (by position), what happens when you `ADD COLUMNS` to the Iceberg table? What's the safe way to read Iceberg data to avoid this problem?

5. **Delta Lake (Databricks) and Apache Hudi are alternatives to Iceberg.** All three solve the same ACID problem on object storage. Research one difference between Iceberg and Delta Lake, and explain which you would recommend for a clinical genomics platform that must run on multiple cloud providers.

*(Write answers here)*

1. 

2. 

3. 

4. 

5. 